In [1]:
import newkernelo as newker
import kernelo as oldker
import numpy as np
import time
import logging
logging.getLogger().setLevel(logging.INFO)

## Global parameters

In [2]:
gamma_type = 'Full'
sigma_type = 'Diag'
seed = 12345678

# initialisation parameters
gllim_em_iteration = 50
gllim_em_floor = 1e-12
gmm_kmeans_iteration = 1
gmm_em_iteration = 1
gmm_floor = 1e-12
nb_experiences = 10

# training parameters
train_max_iteration = 300
train_ratio_ll = 1e-5
train_floor = 1e-12

##### Generate random dataset

In [3]:
L, D, K, N = 4, 9, 5, 1000

x_gen = np.random.rand(N,L)
y_gen = np.random.rand(N,D)

#### Generate sobol Test Model dataset

In [4]:
L, D, K, N = 4, 9, 5, 100

# Create "physical" model
# dir_path = os.path.dirname(os.path.realpath(__file__))
# print(dir_path+"/pytest/models")
# physical_model = oldker.ExternalModelConfig("RPVModel", "rpv_model", dir_path + "/pytest/models").create()
physical_model = oldker.TestModelConfig().create()

# Create StatModel
covariances = np.random.uniform(0, 0.0001, physical_model.get_D_dimension())
stat_model = oldker.GaussianStatModelConfig("sobol", physical_model, covariances, 12345).create()

# Initialize and train GLLIM model
print("Generating dataset")
x_gen, y_gen = stat_model.gen_data(N)
print(x_gen.shape)
print(y_gen.shape)

Generating dataset
(100, 4)
(100, 9)


#### Generate sobol RPV model dataset

In [5]:
L, D, K, N = 3, 71, 10, 100

# Create "physical" model
# dir_path = os.path.dirname(os.path.realpath(__file__))
# print(dir_path+"/pytest/models")
physical_model = oldker.ExternalModelConfig("RPVModel", "rpv_model", "../../pytest/models").create()

# Create StatModel
covariances = np.random.uniform(0, 0.0001, physical_model.get_D_dimension())
stat_model = oldker.GaussianStatModelConfig("sobol", physical_model, covariances, 12345).create()

# Initialize and train GLLIM model
print("Generating dataset")
x_gen, y_gen = stat_model.gen_data(N)
print(x_gen.shape)
print(y_gen.shape)

Generating dataset
(100, 3)
(100, 71)


## OLD GLLiM

In [6]:
# Create GLLIM model, including its initialization and training configuration
learningConfig = oldker.EMLearningConfig(train_max_iteration, train_ratio_ll, train_floor)
# learningConfig=oldker.GMMLearningConfig(train_max_iteration, train_ratio_ll, train_floor)
# initConfig = oldker.MultInitConfig(seed=123456789, nb_iter_EM=10, nb_experiences=10, gmmLearningConfig=oldker.GMMLearningConfig(15, 10, 1e-12))
initConfig = oldker.MultInitConfig(seed=seed, nb_iter_EM=gllim_em_iteration, nb_experiences=nb_experiences, gmmLearningConfig=oldker.GMMLearningConfig(gmm_kmeans_iteration, gmm_em_iteration, gmm_floor))
gllim_old= oldker.GLLiM(D, L, K, gamma_type, sigma_type, initConfig, learningConfig)


In [7]:

print("Initializing GLLIM model")
gllim_old.initialize(x_gen, y_gen)
gllim_old_params_0 = gllim_old.exportModel() # you can export your gllim model parameters

print("Training model")
gllim_old.train(x_gen, y_gen)
gllim_old_params = gllim_old.exportModel() # you can export your gllim model parameters


INFO:root:Start Multi initialization
INFO:root:Initialisation : 1
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 108.208155, Best log likelihood : 108.208155


Initializing GLLIM model


INFO:root:Initialisation : 2
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 102.311355, Best log likelihood : 108.208155
INFO:root:Initialisation : 3
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 121.251314, Best log likelihood : 121.251314
INFO:root:Initialisation : 4
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 107.923949, Best log likelihood : 121.251314
INFO:root:Initialisation : 5
INFO:root:	Generate GMM me

Training model


INFO:root:Iteration : 37, log likelihood : 125.114000
INFO:root:Iteration : 38, log likelihood : 125.114000
INFO:root:Iteration : 39, log likelihood : 125.114000
INFO:root:Iteration : 40, log likelihood : 125.114000
INFO:root:Iteration : 41, log likelihood : 125.114000
INFO:root:Iteration : 42, log likelihood : 125.114000
INFO:root:Iteration : 43, log likelihood : 125.114000
INFO:root:Iteration : 44, log likelihood : 125.114000
INFO:root:Iteration : 45, log likelihood : 125.114000
INFO:root:Iteration : 46, log likelihood : 125.114000
INFO:root:Iteration : 47, log likelihood : 125.114000
INFO:root:Iteration : 48, log likelihood : 125.114000
INFO:root:Iteration : 49, log likelihood : 125.114000
INFO:root:Iteration : 50, log likelihood : 125.114000
INFO:root:Iteration : 51, log likelihood : 125.114000
INFO:root:Iteration : 52, log likelihood : 125.114000
INFO:root:Iteration : 53, log likelihood : 125.114000
INFO:root:Iteration : 54, log likelihood : 125.114000
INFO:root:Iteration : 55, lo

### Get theta_0 from oldker

In [8]:
# theta_0 = newker.GLLiMParameters(L,D,K, "full", "diag")

# print(np.array(gllim_parameters_0.Pi).shape)
# print(np.array(gllim_parameters_0.A).shape)
# print(np.array(gllim_parameters_0.C).shape)
# print(np.array(gllim_parameters_0.Gamma).shape)
# print(np.array(gllim_parameters_0.B).shape)
# print(np.array(gllim_parameters_0.Sigma).shape)

# print(np.array(gllim_parameters_0.Pi))
# print(np.transpose(np.array(gllim_parameters_0.A), (1,2,0)).shape)
# print(np.array(gllim_parameters_0.C).shape)
# print(np.transpose(np.array(gllim_parameters_0.Gamma), (1,2,0)).shape)
# print(np.array(gllim_parameters_0.B).shape)
# print(np.transpose(np.array(gllim_parameters_0.Sigma), (1,2,0)).shape)

# theta_0.Pi = np.copy(np.array(gllim_parameters_0.Pi))
# theta_0.A = np.copy(np.transpose(np.array(gllim_parameters_0.A), (1,2,0)))
# theta_0.C = np.copy(np.array(gllim_parameters_0.C))
# # theta_0.Gamma = np.copy(np.transpose(np.array(gllim_parameters_0.Gamma), (1,2,0)))
# theta_0.Gamma = np.copy(gllim_parameters_0.Gamma)
# theta_0.B = np.copy(np.array(gllim_parameters_0.B))
# # theta_0.Sigma = np.copy(np.transpose(np.array(gllim_parameters_0.Sigma), (1,2,0)))
# Sigma_diag = np.zeros((K,D))
# for k in range(K):
#     Sigma_diag[k,:] = np.diag(gllim_parameters_0.Sigma[k])
# theta_0.Sigma = Sigma_diag

# print(np.sum(theta_0.Pi))
# gllim.setParams(theta_0)

## NEW GLLiM

In [9]:
new_gllim = newker.GLLiM(L,D,K,gamma_type.lower(), sigma_type.lower())

verbose = 0
X_gen = x_gen.T
Y_gen = y_gen.T

new_gllim.initialize(X_gen, Y_gen, gllim_em_iteration, gllim_em_floor, gmm_kmeans_iteration, gmm_em_iteration, gmm_floor, nb_experiences, seed, verbose)
new_gllim_params_0 = new_gllim.getParams()

new_gllim.train(X_gen, Y_gen, train_max_iteration, train_ratio_ll, train_floor, verbose)
new_gllim_params = new_gllim.getParams()


INFO: GLLiM Parameters initialized
INFO: GLLiM dimensions are (L=3, D=71, K=10)
INFO: GLLiM constraints are :
	gamma_type = 'full',
	sigma_type = 'diag'.
0:00:00WARNING: GLLiM model has not been initialized

[]


0:00:02
[]
0:00:03
[[      -inf]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 [0.22305547]
 

In [10]:
print(gllim_old_params_0.Pi)
print(new_gllim_params_0.Pi)

print("\n")
print(gllim_old_params.Pi)
print(new_gllim_params.Pi)

print("\n")
print(gllim_old_params.B)
print(new_gllim_params.B)

[0.07 0.12 0.17 0.04 0.08 0.1  0.11 0.21 0.07 0.03]
[[0.11       0.12       0.07999999 0.08       0.09       0.1
  0.1        0.20000011 0.0799999  0.04      ]]


[0.07 0.12 0.17 0.04 0.08 0.1  0.11 0.21 0.07 0.03]
[[0.11       0.12       0.07999999 0.08       0.09       0.1
  0.1        0.20000011 0.0799999  0.04      ]]


[[ 5.53444862e-01  5.61360267e-01  6.32343337e-01  7.60398580e-01
   9.51438492e-01  1.20075618e+00  1.46498445e+00  1.46498008e+00
   1.20079401e+00  9.51299068e-01]
 [ 7.60430967e-01  6.32268280e-01  5.61377708e-01  5.53395919e-01
   1.46102791e-01  2.18393977e-01  2.85263949e-01  3.66732406e-01
   4.79018427e-01  6.41731691e-01]
 [ 8.77569445e-01  1.20088071e+00  1.58786617e+00  1.87368495e+00
   1.67113193e+00  1.46501838e+00  1.33915141e+00  1.36978051e+00
   3.82754734e+00  3.57479342e+00]
 [ 3.54325357e+00  2.43269699e+00  1.67116948e+00  1.12418382e+00
   7.60436573e-01  5.24278299e-01  3.66925209e-01  2.53508617e-01
   1.60034538e-01  6.46368132e-02]
 [-6.5

In [11]:
# Y_gen = np.array(y_gen[:5].T)
# pred = new_gllim.inverseDensities(Y_gen)
# print(pred.meanPredResult.mean)
# print(x_gen[:5])
# print("\n")


predicator = oldker.PredictionConfig(2, 2, 1e-10, gllim_old).create()
for i in range(5):
    print(x_gen[i])

    prediction = predicator.predict(y_gen[i], np.zeros(D))
    print(prediction.meansPred.mean)

    pred = new_gllim.inverseDensities(y_gen[i])
    print(pred.meanPredResult.mean)
    print("\n")


[0.5 0.5 0.5]
[0.62569596 0.57055746 0.49222032]
[[0.46439839 0.29529598 0.45539489]]


[0.75 0.25 0.25]
[0.75265026 0.20446566 0.2343936 ]
[[0.75529924 0.21617441 0.23706769]]


[0.25 0.75 0.75]
[0.84973618 0.75389486 0.89972216]
[[0.19434935 0.77125522 0.62278123]]


[0.375 0.375 0.625]
[0.3647228  0.35873891 0.62201617]
[[0.50375196 0.18917834 0.71163335]]


[0.875 0.875 0.125]
[0.56753417 0.95323174 0.14179398]
[[0.56753417 0.95323174 0.14179398]]




In [12]:
pred.centerPredResult.means

array([], shape=(0, 0), dtype=float64)

In [13]:
insights = new_gllim.getInsights()

In [14]:
print(insights.time)

0:00:03
